### BulkFormer feature extraction

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  

In [2]:
import math
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import pearsonr, spearmanr
from collections import OrderedDict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset,DataLoader,random_split
from torch_geometric.typing import SparseTensor

In [3]:
from utils.BulkFormer import BulkFormer

In [4]:
from model.config import model_params

In [5]:
device = 'cuda'

In [6]:
graph_path = 'data/G_gtex.pt'
weights_path = 'data/G_gtex_weight.pt'
gene_emb_path = 'data/esm2_feature_concat.pt'

In [7]:
graph = torch.load(graph_path, map_location='cpu', weights_only=False)
weights = torch.load(weights_path, map_location='cpu', weights_only=False)
graph = SparseTensor(row=graph[1], col=graph[0], value=weights).t().to(device)
gene_emb = torch.load(gene_emb_path, map_location='cpu', weights_only=False)
model_params['graph'] = graph
model_params['gene_emb'] = gene_emb

In [8]:
model = BulkFormer(**model_params).to(device)

In [9]:
ckpt_model = torch.load('model/Bulkformer_ckpt_epoch_29.pt',weights_only=False)

In [10]:
# strips blocks that start with module, not sure why
new_state_dict = OrderedDict()
for key, value in ckpt_model.items():
    new_key = key[7:] if key.startswith("module.") else key
    new_state_dict[new_key] = value

In [11]:
model.load_state_dict(new_state_dict)

<All keys matched successfully>

In [39]:
def extract_feature(expr_array, 
                    high_var_gene_idx,
                    feature_type,
                    aggregate_type,
                    device,
                    batch_size,
                    return_expr_value = False,
                    esm2_emb = None,
                    valid_gene_idx = None):

    expr_tensor = torch.tensor(expr_array,dtype=torch.float32,device=device)
    mydataset = TensorDataset(expr_tensor)
    myloader = DataLoader(mydataset, batch_size=batch_size, shuffle=False) 
    model.eval()

    all_emb_list = []
    all_expr_value_list = []


    with torch.no_grad():
        if feature_type == 'transcriptome_level':
            for (X,) in tqdm(myloader, total=len(myloader)):
                X = X.to(device)
                output, emb = model(X, [2])
                # output shape [batch, n_genes]
                # emb shape [batch, n_genes, proj_dim]

                all_expr_value_list.append(output.detach().cpu().numpy())
                emb = emb[2].detach().cpu().numpy()
                emb_valid = emb[:,high_var_gene_idx,:]
     
                if aggregate_type == 'max':
                    final_emb =np.max(emb_valid, axis=1)
                elif aggregate_type == 'mean':
                    final_emb =np.mean(emb_valid, axis=1)
                elif aggregate_type == 'median':
                    final_emb =np.median(emb_valid, axis=1)
                elif aggregate_type == 'all':
                    max_emb =np.max(emb_valid, axis=1)
                    mean_emb =np.mean(emb_valid, axis=1)
                    median_emb =np.median(emb_valid, axis=1)
                    final_emb = max_emb+mean_emb+median_emb

                all_emb_list.append(final_emb)
            result_emb = np.vstack(all_emb_list)
            result_emb = torch.tensor(result_emb,device='cpu',dtype=torch.float32)

        elif feature_type == 'gene_level':
            for (X,) in tqdm(myloader, total=len(myloader)):
                X = X.to(device)
                output, emb = model(X, [2])
                emb = emb[2].detach().cpu().numpy()
                emb_valid = emb[:,valid_gene_idx,:]
                all_emb_list.append(emb_valid)
                all_expr_value_list.append(output.detach().cpu().numpy())
            all_emb = np.vstack(all_emb_list)
            all_emb_tensor = torch.tensor(all_emb,device='cpu',dtype=torch.float32)
            esm2_emb_selected = esm2_emb[valid_gene_idx]
            esm2_emb_expanded = esm2_emb_selected.unsqueeze(0).expand(all_emb_tensor.shape[0], -1, -1)  # [B, N, D]
            esm2_emb_expanded = esm2_emb_expanded.to('cpu')

            result_emb = torch.cat([all_emb_tensor, esm2_emb_expanded], dim=-1)
    
    if return_expr_value:
        return np.vstack(all_expr_value_list)
    
    else:
        return result_emb

In [31]:
def main_gene_selection(X_df, gene_list):
    # fills columns (genes) that are in the gene list but not in our data
    # with -10
    # Returns df containing gene_list values (with -10 filling), columns that
    # were filled, and var, df indicating which columns are masked (to fill)

    to_fill_columns = list(set(gene_list) - set(X_df.columns))


    padding_df = pd.DataFrame(np.full((X_df.shape[0], len(to_fill_columns)), -10), 
                            columns=to_fill_columns, 
                            index=X_df.index)

    X_df = pd.DataFrame(np.concatenate([df.values for df in [X_df, padding_df]], axis=1), 
                        index=X_df.index, 
                        columns=list(X_df.columns) + list(padding_df.columns))
    X_df = X_df[gene_list]
    
    var = pd.DataFrame(index=X_df.columns)
    var['mask'] = [1 if i in to_fill_columns else 0 for i in list(var.index)]
    return X_df, to_fill_columns,var

In [32]:
# load demo data
# demo_df = pd.read_csv('data/demo.csv')
demo_df = pd.read_parquet("../UCLThesis/data/BIOAID_UCL_Oxford_361_combined.parquet")
demo_df = demo_df.loc[:, "5S_rRNA":]
# demo_df

In [33]:
bulkformer_gene_info = pd.read_csv('data/bulkformer_gene_info.csv')
# fix extra row?
bulkformer_gene_info = bulkformer_gene_info[bulkformer_gene_info['ensg_id'] != '35991']

In [34]:
# bulkformer_gene_list = bulkformer_gene_info['ensg_id'].to_list()
# Use gene symbols to match with ucl data
bulkformer_gene_list = list(bulkformer_gene_info["gene_symbol"])

In [35]:

# input_df , to_fill_columns, var= main_gene_selection(X_df=demo_df,gene_list=bulkformer_gene_list)
# ucl_gene_data = pd.read_parquet("../UCLThesis/data/BIOAID_UCL_Oxford_361_combined.parquet")
input_df , to_fill_columns, var= main_gene_selection(X_df=demo_df,gene_list=bulkformer_gene_list)

In [36]:
var.reset_index(inplace=True)
valid_gene_idx = list(var[var['mask'] == 0].index)

In [37]:
high_var_gene_idx = torch.load('data/high_var_gene_list.pt',weights_only=False)
len(high_var_gene_idx)

2000

In [38]:
# Extract transcritome-level embedding
result = extract_feature(
    expr_array= input_df[:16].values,
    high_var_gene_idx=high_var_gene_idx,
    feature_type='transcriptome_level',
    aggregate_type='max',
    device=device,
    batch_size=4,
    return_expr_value=False,
    esm2_emb=model_params['gene_emb'],
    valid_gene_idx=valid_gene_idx
)

 25%|██▌       | 1/4 [00:00<00:02,  1.34it/s]

tensor([48560.2773, 49328.3750, 51046.1602, 48307.3398], device='cuda:0')


 50%|█████     | 2/4 [00:01<00:01,  1.34it/s]

tensor([53406.2344, 53447.1172, 53481.6172, 53982.2266], device='cuda:0')


 75%|███████▌  | 3/4 [00:02<00:00,  1.33it/s]

tensor([55209.0820, 47289.5312, 55079.0156, 53202.6562], device='cuda:0')


100%|██████████| 4/4 [00:02<00:00,  1.34it/s]

tensor([55048.4062, 50055.3516, 50459.1250, 52168.5000], device='cuda:0')


In [21]:
result.shape

torch.Size([16, 640])

In [22]:
# save embeddings
ucl_embeddings = pd.DataFrame(result.numpy(), columns=[f"col_{i}" for i in range(640)])
ucl_embeddings.to_parquet("../UCLThesis/data/BIOAID_361_embeddings.parquet", index=False)

In [77]:
# Extract gene-level embedding
result = extract_feature(
    expr_array= input_df.values[:16],
    high_var_gene_idx=high_var_gene_idx,
    feature_type='gene_level',
    aggregate_type='all',
    device=device,
    batch_size=4,
    return_expr_value=False,
    esm2_emb=model_params['gene_emb'],
    valid_gene_idx=valid_gene_idx
)

  0%|          | 0/4 [00:00<?, ?it/s]


AttributeError: 'dict' object has no attribute 'shape'

In [24]:
result.shape

torch.Size([16, 20010, 1920])

In [25]:
result

tensor([[[-0.9206,  0.0835, -0.9465,  ..., -0.0972, -0.1156, -0.0694],
         [-0.9525,  0.0228, -1.0047,  ..., -0.0929, -0.0102,  0.0749],
         [-1.3258, -0.2688, -1.0380,  ..., -0.1507, -0.0174,  0.1455],
         ...,
         [-0.9440,  0.0216, -1.0109,  ..., -0.0316,  0.0078,  0.0943],
         [-0.9463,  0.0198, -1.0248,  ..., -0.0891, -0.0469,  0.1897],
         [-0.9317,  0.0270, -1.0059,  ..., -0.0528, -0.0946,  0.0670]],

        [[-0.5924, -0.9445, -0.3474,  ..., -0.0972, -0.1156, -0.0694],
         [-0.3951, -0.9887, -0.9957,  ..., -0.0929, -0.0102,  0.0749],
         [-1.3602, -1.3107, -1.3065,  ..., -0.1507, -0.0174,  0.1455],
         ...,
         [-1.0295, -1.0003, -0.2994,  ..., -0.0316,  0.0078,  0.0943],
         [-0.3802, -0.9847, -1.0505,  ..., -0.0891, -0.0469,  0.1897],
         [-0.3836, -0.9666, -1.0262,  ..., -0.0528, -0.0946,  0.0670]],

        [[-0.6962, -0.2097, -0.9280,  ..., -0.0972, -0.1156, -0.0694],
         [-0.7126, -0.2782, -1.0050,  ..., -0

In [26]:
# Extract expression values
result = extract_feature(
    expr_array= input_df.values[:16],
    high_var_gene_idx=high_var_gene_idx,
    feature_type='transcriptome_level',
    aggregate_type='all',
    device=device,
    batch_size=4,
    return_expr_value=True,
    esm2_emb=model_params['gene_emb'],
    valid_gene_idx=valid_gene_idx
)

100%|██████████| 4/4 [00:03<00:00,  1.24it/s]


In [27]:
result.shape

(16, 20010)

In [28]:
ucl_gene_data = pd.read_parquet("../UCLThesis/data/BIOAID_UCL_Oxford_361_combined.parquet")

In [29]:
# consider trying to match on ensembl ids instead next.
ucl_gene_list = list(ucl_gene_data.loc[:, "5S_rRNA":].columns)
print(ucl_gene_list[:4])
len(ucl_gene_list)

['5S_rRNA', 'A1BG', 'A1CF', 'A2M']


29652

In [30]:
bulkformer_gene_list = list(bulkformer_gene_info['gene_symbol'])
len(bulkformer_gene_list)

20010

In [31]:
ucl_gene_set = set(ucl_gene_list)
bulkformer_gene_set = set(bulkformer_gene_list)

In [32]:
common_genes = ucl_gene_set.intersection(bulkformer_gene_set)
len(common_genes)

19338

In [33]:
ucl_gene_set.difference(bulkformer_gene_set)

{'IGKV1D-39',
 'RHOQP3',
 'ARF1P2',
 'CA15P1',
 'RPL23AP26',
 'SCOCP1',
 'BNIP3P20',
 'RBBP4P1',
 'NPM1P5',
 'MTATP6P27',
 'GAPDHP26',
 'ELOCP30',
 'RNA5SP521',
 'FAM86JP',
 'MTATP6P17',
 'MTCO3P20',
 'OR5H4P',
 'RPL14P6',
 'HERC2P11',
 'DPY19L2P2',
 'COMMD5P1',
 'PIGFP1',
 'SLC44A3-AS1',
 'TAS2R45',
 'ATP6AP1L',
 'PPY2P',
 'RPL23AP58',
 'NATP',
 'DGKZP1',
 'IFIT1P1',
 'COX5BP2',
 'FUNDC2P4',
 'NCAPD2P1',
 'EMBP1',
 'RNA5SP349',
 'RPS27AP6',
 'HLA-P',
 'MTND3P18',
 'METAP2P1',
 'PTGES3P2',
 'MRPL40P1',
 'HNRNPCP1',
 'RPL7AP9',
 'STMN1P2',
 'TPT1P2',
 'RNA5SP385',
 'ACTBP4',
 'RNA5SP301',
 'HNRNPA1P16',
 'MRPS17P5',
 'MED15P9',
 'EIF1P1',
 'RPS26P24',
 'KRT8P20',
 'IGHG3',
 'ZSWIM5P3',
 'RDXP1',
 'SLC31A1P1',
 'RNA5SP99',
 'OR2AT1P',
 'TAS2R6P',
 'HMGN2P27',
 'RPS12P3',
 'BTBD10P2',
 'RPL7AP4',
 'MTND1P2',
 'MTCO3P12',
 'AMYP1',
 'RPL5P25',
 'TOMM20P1',
 'ANXA2P2',
 'SLC35E2A',
 'RPS15P9',
 'ST13P12',
 'RCC2P5',
 'RBM22P1',
 'MTCO1P11',
 'FABP5P10',
 'RPL10P15',
 'RPS3AP47',
 'PPIAP38',

In [34]:
bulkformer_gene_set.difference(ucl_gene_set)

{'ARNTL',
 'ARNTL2',
 'BHLHB9',
 'BTBD11',
 'C10orf99',
 'C11orf53',
 'C16orf72',
 'C17orf64',
 'C19orf71',
 'C7orf61',
 'CBWD1',
 'CBWD2',
 'CBWD3',
 'CBWD5',
 'CBWD6',
 'COLCA2',
 'CYHR1',
 'DDX58',
 'ENSG00000100101',
 'ENSG00000111780',
 'ENSG00000124593',
 'ENSG00000125695',
 'ENSG00000131152',
 'ENSG00000141979',
 'ENSG00000142539',
 'ENSG00000144785',
 'ENSG00000159239',
 'ENSG00000167774',
 'ENSG00000167807',
 'ENSG00000170846',
 'ENSG00000173366',
 'ENSG00000173867',
 'ENSG00000183889',
 'ENSG00000187186',
 'ENSG00000188223',
 'ENSG00000188897',
 'ENSG00000196826',
 'ENSG00000197991',
 'ENSG00000198211',
 'ENSG00000203546',
 'ENSG00000204003',
 'ENSG00000205236',
 'ENSG00000206549',
 'ENSG00000213204',
 'ENSG00000214265',
 'ENSG00000214558',
 'ENSG00000225528',
 'ENSG00000226490',
 'ENSG00000226690',
 'ENSG00000228144',
 'ENSG00000230707',
 'ENSG00000233757',
 'ENSG00000235007',
 'ENSG00000236543',
 'ENSG00000237378',
 'ENSG00000239395',
 'ENSG00000239920',
 'ENSG00000241489',